# Federated Learning Experiments - Google Colab Setup

This notebook sets up the federated learning framework and runs experiments on the EBHI-SEG dataset.

**Features:**
- Auto-download dataset from figshare
- Compare FL algorithms: FedAvg, FedProx, FedOptimizer
- Compare models: UNet, DeepLabV3, FCN
- Save results to Google Drive

**GPU:** Uses Colab's free T4 GPU (auto-selected)

## Step 1: Mount Google Drive & Setup

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive'

# Create project directory in Drive
PROJECT_DIR = os.path.join(DRIVE_PATH, 'federated_learning')
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, 'data'), exist_ok=True)

print(f'✓ Google Drive mounted')
print(f'✓ Project directory: {PROJECT_DIR}')

## Step 2: Clone Repository

In [ ]:
import subprocess
import os
from getpass import getpass

# Clone the private repository to local Colab storage (fast I/O)
repo_path = '/content/federated_learning_repo'

if not os.path.exists(repo_path):
    print('Cloning private repository...\n')
    print('You will need a GitHub Personal Access Token (PAT) for private repo access.\n')
    print('To create a PAT:')
    print('  1. Go to https://github.com/settings/tokens')
    print('  2. Click "Generate new token (classic)"')
    print('  3. Enable "repo" scope')
    print('  4. Copy the token below\n')
    
    github_token = getpass('Enter your GitHub Personal Access Token: ')
    github_username = input('Enter your GitHub username: ')
    repo_name = input('Enter repository name (e.g., federated_learning): ')
    
    clone_url = f'https://{github_token}@github.com/{github_username}/{repo_name}.git'
    
    try:
        subprocess.run([
            'git', 'clone',
            clone_url,
            repo_path
        ], check=True, capture_output=True)
        print('\n✓ Repository cloned successfully')
    except subprocess.CalledProcessError as e:
        print(f'\n✗ Clone failed: {e.stderr.decode()}')
        print('\nAlternative: Upload repo as ZIP to Drive')
        print('  1. On your machine: zip -r federated_learning.zip federated_learning/')
        print('  2. Upload ZIP to MyDrive/federated_learning.zip')
        print('  3. Re-run this cell and choose the ZIP option')
        raise
else:
    print('✓ Repository already exists')

# Add to Python path
import sys
sys.path.insert(0, repo_path)
os.chdir(repo_path)

print(f'✓ Working directory: {repo_path}')


## Step 3: Install Dependencies

In [ ]:
import subprocess
import sys

print('Installing dependencies...')

# Core dependencies
packages = [
    'torch>=2.0',
    'torchvision',
    'pyyaml',
    'pillow',
    'numpy',
    'tqdm',
    'matplotlib',
    'scikit-learn',
    'albumentations',
    'requests',  # for downloading dataset
]

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, check=True)

print('✓ All dependencies installed')

## Step 4: Download Dataset

In [ ]:
import os
import shutil

# Paths
drive_dataset = os.path.join(DRIVE_PATH, 'EBHI-SEG')
data_dir = '/content/data/EBHI-SEG'

print('=' * 70)
print('DATASET SETUP')
print('=' * 70)

# Copy from Drive to local Colab storage
if not os.path.exists(data_dir):
    os.makedirs('/content/data', exist_ok=True)
    if os.path.exists(drive_dataset):
        print(f'Copying from: {drive_dataset}')
        shutil.copytree(drive_dataset, data_dir)
        print(f'✓ Dataset copied to: {data_dir}\n')
    else:
        raise FileNotFoundError(f"Dataset not found at {drive_dataset}")
else:
    print(f'Dataset already in local storage\n')

# Count and display images
print('Dataset structure:\n')
categories = sorted([d for d in os.listdir(data_dir) 
                    if os.path.isdir(os.path.join(data_dir, d)) 
                    and not d.startswith('.')])

total_images = 0
for category in categories:
    image_dir = os.path.join(data_dir, category, 'image')
    if os.path.exists(image_dir):
        count = len([f for f in os.listdir(image_dir) if os.path.isfile(os.path.join(image_dir, f))])
        total_images += count
        print(f'  {category:20} {count:>4} images')

print(f'\n{"="*70}')
print(f'Total: {total_images} images')
print(f'✓ Ready for training!')


In [ ]:
import yaml

# Update config to use the local data path
config_path = os.path.join(repo_path, 'configs', 'config.yaml')

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

config['data_root'] = '/content/data/EBHI-SEG'

with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'✓ Updated config.yaml: data_root = {config["data_root"]}')


## Step 5: Verify Setup

In [ ]:
import torch
from src.model import create_model

print('Checking setup...')
print(f'✓ PyTorch version: {torch.__version__}')
print(f'✓ GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')

# Test models load
print(f'\n✓ Testing model factory...')
for model_type in ['unet', 'deeplab', 'fcn']:
    m = create_model(model_type, 3, 1)
    params = sum(p.numel() for p in m.parameters())
    print(f'  {model_type.upper():10} - {params:>10,} params')

print(f'\n✓ All systems ready!')

## Step 6: Run Baseline (Centralized Training)

In [ ]:
# Run centralized training baseline (UNet, 30 epochs)
# This establishes a baseline to compare federated learning against
print('Running centralized training baseline with UNet...')
print('This will train on the full dataset (no federation)\n')

!python main.py \
  --mode centralized \
  --model_type unet \
  --epochs 30 \
  --batch_size 16 \
  --device cuda \
  --seed 42

## Step 7: Comprehensive Experiment - Models × Algorithms × Distributions

In [ ]:
# Comprehensive experiment: all combinations of models, algorithms, and distributions
# Each result stored in separate directory: outputs/[model]/[algorithm]/[distribution]/

import os
import json

models = ['unet', 'deeplab', 'fcn']
algorithms = ['fedavg', 'fedprox', 'fedoptimizer']
distributions = [
    ('by_class', 'Non-IID'),
    ('random', 'IID'),
]

total_experiments = len(models) * len(algorithms) * len(distributions)
current = 0

results_summary = []

for model in models:
    for algo in algorithms:
        for dist_strategy, dist_name in distributions:
            current += 1
            
            # Create unique output directory for this experiment
            exp_name = f"{model}_{algo}_{dist_strategy}"
            exp_dir = os.path.join(repo_path, 'outputs', exp_name)
            os.makedirs(exp_dir, exist_ok=True)
            
            print(f'\n{"="*70}')
            print(f'[{current}/{total_experiments}] {model.upper()} + {algo.upper()} + {dist_name}')
            print(f'{"="*70}')
            print(f'Output: {exp_dir}\n')
            
            # Run experiment
            !python main.py \
              --mode federated \
              --model_type {model} \
              --fl_algorithm {algo} \
              --partition {dist_strategy} \
              --n_clients 6 \
              --fl_rounds 20 \
              --local_epochs 3 \
              --batch_size 16 \
              --device cuda \
              --output_dir {exp_dir} \
              --seed 42
            
            # Store summary
            results_summary.append({
                'experiment': exp_name,
                'model': model,
                'algorithm': algo,
                'distribution': dist_name,
                'output_dir': exp_dir
            })

# Save experiment summary
summary_file = os.path.join(repo_path, 'outputs', 'EXPERIMENT_SUMMARY.json')
with open(summary_file, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f'\n{"="*70}')
print(f'All {total_experiments} experiments completed!')
print(f'Summary saved to: {summary_file}')
print(f'{"="*70}')


## Step 8: Load and Visualize Results

In [ ]:
import json
import os
from pathlib import Path

# Load and display results from all experiments
outputs_base = Path('outputs')
summary_file = outputs_base / 'EXPERIMENT_SUMMARY.json'

print('Experiment Results Summary:')
print('='*70)

if summary_file.exists():
    with open(summary_file) as f:
        experiments = json.load(f)
    
    print(f'\nTotal experiments: {len(experiments)}\n')
    
    for exp in experiments:
        exp_dir = Path(exp['output_dir'])
        history_file = exp_dir / 'fl_history.json'
        
        if history_file.exists():
            with open(history_file) as f:
                history = json.load(f)
            
            if 'rounds' in history and len(history['rounds']) > 0:
                latest = history['rounds'][-1]
                print(f"✓ {exp['experiment']}")
                print(f"  Model: {exp['model'].upper():10} | Algo: {exp['algorithm'].upper():12} | Dist: {exp['distribution']:7}")
                print(f"  Dice: {latest.get('dice', 0):.4f} | IoU: {latest.get('iou', 0):.4f} | Loss: {latest.get('loss', 0):.4f}")
                print()

print('='*70)
print('Full results in outputs/[model]_[algorithm]_[distribution]/')


## Step 9: Copy Results to Google Drive

In [ ]:
import shutil
from pathlib import Path

# Copy all results to Drive for safekeeping
src = Path('outputs')
dst = Path(DRIVE_PATH) / 'federated_learning_results'

if src.exists():
    print(f'Syncing results to Google Drive...')
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'✓ Results saved to: {dst}')
    
    # List experiment dirs
    print(f'\nExperiment directories:')
    for d in sorted(dst.glob('*')):
        if d.is_dir() and not d.name.startswith('.'):
            print(f'  {d.name}/')
else:
    print('No results directory found')


## Summary

You've successfully run a comprehensive federated learning experiment:

**18 Total Experiments:**
- **3 Models:** UNet, DeepLabV3, FCN
- **3 Algorithms:** FedAvg, FedProx, FedOptimizer
- **2 Distributions:** Non-IID (by_class), IID (random)

Each experiment stored separately in:
```
outputs/
  unet_fedavg_by_class/
  unet_fedavg_random/
  unet_fedprox_by_class/
  ... (18 total)
  EXPERIMENT_SUMMARY.json
```

### Next Steps:
- Download results from Google Drive
- Compare metrics (Dice, IoU) across models/algorithms/distributions
- Analyze convergence curves
- Generate comparison plots

### Troubleshooting:
- **Out of memory:** Reduce `batch_size` or `fl_rounds`
- **Session timeout:** Experiments already run don't re-run (check outputs/)
- **Network issues:** Results are synced to Drive automatically
